## Setup

In [1]:
# notebooks/02_statistical_summary.ipynb

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
from scipy import stats
from src.data_loader import DataLoader

PROJECT_ROOT = Path.cwd().parent
reports_dir = PROJECT_ROOT / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)

# steamId is an identifier, not a real variable — exclude it from all
# numeric analysis below (summary stats, normality tests, correlations).
ID_COLUMNS = ['steamId']


def numeric_columns(df: pd.DataFrame) -> pd.Index:
    cols = df.select_dtypes(include=['int64', 'float64']).columns
    return cols.difference(ID_COLUMNS)


## Load Cleaned Data

In [2]:
# Load cleaned data
loader = DataLoader()
df = pd.read_csv(loader.processed_dir / "cleaned_data.csv")


## 1. Summary Statistics

In [3]:
def comprehensive_stats(df: pd.DataFrame) -> pd.DataFrame:
    """
    Generate comprehensive statistics for all numeric columns (excluding IDs).
    """
    numeric_cols = numeric_columns(df)

    stats_dict = {
        'Column': [], 'Mean': [], 'Median': [], 'Mode': [], 'Std': [],
        'Variance': [], 'Skewness': [], 'Kurtosis': [], 'Range': [],
        'IQR': [], 'Q1': [], 'Q3': [], 'CV': [], 'Missing %': []
    }

    for col in numeric_cols:
        data = df[col].dropna()

        stats_dict['Column'].append(col)
        stats_dict['Mean'].append(data.mean())
        stats_dict['Median'].append(data.median())
        stats_dict['Mode'].append(data.mode()[0] if not data.mode().empty else np.nan)
        stats_dict['Std'].append(data.std())
        stats_dict['Variance'].append(data.var())
        stats_dict['Skewness'].append(data.skew())
        stats_dict['Kurtosis'].append(data.kurtosis())
        stats_dict['Range'].append(data.max() - data.min())
        stats_dict['IQR'].append(data.quantile(0.75) - data.quantile(0.25))
        stats_dict['Q1'].append(data.quantile(0.25))
        stats_dict['Q3'].append(data.quantile(0.75))
        stats_dict['CV'].append(data.std() / data.mean() if data.mean() != 0 else np.nan)
        stats_dict['Missing %'].append(df[col].isnull().mean() * 100)

    return pd.DataFrame(stats_dict)


In [4]:
stats_df = comprehensive_stats(df)
stats_df.to_csv(reports_dir / 'summary_statistics.csv', index=False)
print("Summary Statistics:")
print(stats_df.to_string())


Summary Statistics:
             Column          Mean         Median         Mode           Std      Variance   Skewness    Kurtosis         Range            IQR            Q1             Q3         CV  Missing %
0       avgPlaytime  1.256270e+01       6.762776      0.00000  2.154217e+01  4.640652e+02   7.170166   71.789873  2.963329e+02       9.539626      3.564848      13.104473   1.714772        0.0
1        copiesSold  1.414826e+05   11928.500000   3127.00000  1.132757e+06  1.283138e+12  18.619478  421.415821  3.073856e+07   32951.000000   4918.750000   37869.750000   8.006334        0.0
2             price  1.751951e+01      14.990000     19.99000  1.264661e+01  1.599368e+02   1.576667    3.870984  9.999000e+01      10.000000      9.990000      19.990000   0.721859        0.0
3           revenue  2.632382e+06  109053.000000  21939.00000  2.781024e+07  7.734094e+14  22.920076  609.817926  8.377727e+08  409652.500000  45504.250000  455156.750000  10.564667        0.0
4  revenue_per_

## 2. Normality Tests

In [5]:
def test_normality(df: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    """
    Perform Shapiro-Wilk test for normality on numeric columns (excluding IDs).

    NOTE: Shapiro-Wilk gets more sensitive as sample size grows. With
    n≈1500, it will likely flag most columns as "Not normal" — including
    ones that are only mildly skewed — because it can detect small,
    practically unimportant departures from normality. Treat the p-value
    as one signal, not the final word; a histogram or QQ-plot per column
    is a useful sanity check before deciding which correlation method
    (Pearson vs. Spearman) or test to rely on.
    """
    numeric_cols = numeric_columns(df)

    results = {'Column': [], 'Statistic': [], 'P-value': [], 'Normal?': [], 'Interpretation': []}

    for col in numeric_cols:
        data = df[col].dropna()
        if len(data) > 3:
            stat, p_value = stats.shapiro(data)
            is_normal = p_value > alpha

            results['Column'].append(col)
            results['Statistic'].append(stat)
            results['P-value'].append(p_value)
            results['Normal?'].append(is_normal)
            results['Interpretation'].append("Normal" if is_normal else "Not normal")

    return pd.DataFrame(results)


In [6]:
normality_df = test_normality(df)
normality_df.to_csv(reports_dir / 'normality_tests.csv', index=False)
print("\nNormality Tests:")
print(normality_df.to_string())



Normality Tests:
             Column  Statistic       P-value  Normal? Interpretation
0       avgPlaytime   0.434166  5.493773e-56    False     Not normal
1        copiesSold   0.086137  1.431986e-64    False     Not normal
2             price   0.881524  2.318088e-32    False     Not normal
3           revenue   0.058936  4.076591e-65    False     Not normal
4  revenue_per_copy   0.724465  2.775911e-44    False     Not normal
5       reviewScore   0.734605  1.033622e-43    False     Not normal


## 3. Correlation Analysis

In [7]:
def correlation_analysis(df: pd.DataFrame) -> dict:
    """
    Calculate correlation matrices using different methods (excluding IDs).
    """
    numeric_cols = numeric_columns(df)
    corr_matrix = df[numeric_cols].corr()
    spearman_matrix = df[numeric_cols].corr(method='spearman')

    corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i + 1, len(corr_matrix.columns)):
            corr_value = corr_matrix.iloc[i, j]
            corr_pairs.append({
                'Variable 1': corr_matrix.columns[i],
                'Variable 2': corr_matrix.columns[j],
                'Correlation': corr_value,
                'Strength': abs(corr_value)
            })

    corr_pairs = sorted(corr_pairs, key=lambda x: x['Strength'], reverse=True)

    return {
        'pearson': corr_matrix,
        'spearman': spearman_matrix,
        'top_correlations': corr_pairs[:10]
    }


In [8]:
corr_results = correlation_analysis(df)
corr_results['pearson'].to_csv(reports_dir / 'pearson_correlation.csv')
corr_results['spearman'].to_csv(reports_dir / 'spearman_correlation.csv')

print("\nTop 5 Correlations:")
for i, pair in enumerate(corr_results['top_correlations'][:5], 1):
    print(f"{i}. {pair['Variable 1']} <-> {pair['Variable 2']}: {pair['Correlation']:.3f}")



Top 5 Correlations:
1. price <-> revenue_per_copy: 0.753
2. copiesSold <-> revenue: 0.628
3. avgPlaytime <-> revenue_per_copy: 0.191
4. price <-> revenue: 0.163
5. revenue <-> revenue_per_copy: 0.157
